# GEOM7001 Model Analysis and Visualisation

## Multi-sensor EO Prediction of Mine Rehabilitation

This notebook presents the post-model analysis and visualisation of the Random Forest prediction experiments.

The modelling workflow compares Sentinel-2 optical predictors with a combined Sentinel-2 and Sentinel-1 predictor set for forecasting Fractional Vegetation Cover (FVC) at 1-year and 5-year prediction horizons.

The preprocessing and primary model validation were completed separately. This notebook uses the harmonised 30 m monthly dataset and previously saved validation results to produce additional model diagnostics, spatial visualisations and feature-importance analysis.

## 1. Setup

The analysis uses the harmonised monthly modelling dataset covering January 2016 to December 2025. The dataset contains three Sentinel-2 vegetation indices (NDVI, EVI and NDRE), two Sentinel-1 SAR backscatter variables (VV and VH), and Fractional Vegetation Cover (FVC).

Previously saved spatial cross-validation results are also loaded so that the completed model validation does not need to be repeated.

In [1]:
import json
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr

from shapely.geometry import shape
from affine import Affine
from rasterio.features import geometry_mask

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from pathlib import Path

print("Saved models:")
for f in Path(".").rglob("*.joblib"):
    print(f)

print("\nPrediction files:")
for f in Path(".").rglob("*prediction*.csv"):
    print(f)

import joblib

model = joblib.load("rf_1year_fold1.joblib")

print("Model type:", type(model))
print("Number of features:", model.n_features_in_)

if hasattr(model, "feature_names_in_"):
    print("Features:", model.feature_names_in_)

Saved models:
rf_1year_fold1.joblib
rf_5year_sar.joblib
rf_1year_sar_importance.joblib

Prediction files:
predictions_1year_test.csv
predictions_5year_test.csv
Model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Number of features: 5
Features: ['NDVI' 'EVI' 'NDRE' 'VV' 'VH']


### Load the Harmonised Modelling Dataset

The preprocessed modelling dataset is loaded directly from disk. All predictor and target variables have already been aligned to the same 30 m spatial grid and monthly temporal resolution.

In [23]:
import joblib

model = joblib.load("rf_1year_fold1.joblib")

print(model.get_params())

{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [ ]:
saved = pd.read_csv("predictions_1year_test.csv")

keys = ["x", "y", "predictor_date", "target_date"]

print("Saved CSV rows:", len(saved))
print("Original samples:", len(samples_1yr))

print("\nDuplicate keys:")
print("Saved:", saved.duplicated(keys).sum())
print("Samples:", samples_1yr.duplicated(keys).sum())

print("\nDate examples:")
print("Saved:", saved["predictor_date"].head(3).tolist())
print("Samples:", samples_1yr["predictor_date"].head(3).tolist())

print("\nSaved test blocks:", saved["block_id"].nunique())
# The original test_idx is not defined at this stage.
# Test-block comparison is performed after spatial folds are reconstructed below.

Saved CSV rows: 518615
Original samples: 2431453

Duplicate keys:
Saved: 0
Samples: 0

Date examples:
Saved: ['2016-01-01', '2016-01-01', '2016-01-01']
Samples: [Timestamp('2016-01-01 00:00:00'), Timestamp('2016-01-01 00:00:00'), Timestamp('2016-01-01 00:00:00')]

Saved test blocks: 21


In [ ]:
saved_blocks = set(saved["block_id"])
current_blocks = set(samples_1yr.iloc[test_idx]["block_id"])

print("Same test blocks:", saved_blocks == current_blocks)

print("Blocks only in saved CSV:")
print(saved_blocks - current_blocks)

print("\nBlocks only in current test:")
print(current_blocks - saved_blocks)

# Check whether saved test blocks exist in the original dataset
original_test = samples_1yr[
    samples_1yr["block_id"].isin(saved_blocks)
].copy()

print("\nRecovered test rows:", len(original_test))
print("Saved CSV rows:", len(saved))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Recover original test blocks
saved_blocks = set(saved["block_id"])

train_data = samples_1yr[
    ~samples_1yr["block_id"].isin(saved_blocks)
].copy()

test_data = samples_1yr[
    samples_1yr["block_id"].isin(saved_blocks)
].copy()

# 2. Select optical features
features = ["NDVI", "EVI", "NDRE"]

X_train = train_data[features]
y_train = train_data["FVC_target"]

X_test = test_data[features]
y_test = test_data["FVC_target"]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 3. Train optical-only Random Forest
rf_1year_optical = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_1year_optical.fit(X_train, y_train)

# 4. Predict
y_pred = rf_1year_optical.predict(X_test)

# 5. Evaluate
print("\n1-Year Optical-only")
print("MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 3))
print("R²:", round(r2_score(y_test, y_pred), 3))

# 6. Save model
joblib.dump(rf_1year_optical, "rf_1year_optical.joblib")

In [13]:
model_data = xr.open_dataset(
    "model_data_2016_2025.nc"
)

print(model_data)

<xarray.Dataset> Size: 149MB
Dimensions:      (time: 120, y: 216, x: 180)
Coordinates:
  * time         (time) datetime64[ns] 960B 2016-01-01 2016-02-01 ... 2025-12-01
  * x            (x) float64 1kB 3.701e+05 3.701e+05 ... 3.754e+05 3.754e+05
  * y            (y) float64 2kB 6.984e+06 6.984e+06 ... 6.977e+06 6.977e+06
Data variables:
    NDVI         (time, y, x) float32 19MB ...
    EVI          (time, y, x) float32 19MB ...
    NDRE         (time, y, x) float32 19MB ...
    VV           (time, y, x) float64 37MB ...
    VH           (time, y, x) float64 37MB ...
    FVC          (time, y, x) float32 19MB ...
    spatial_ref  int32 4B ...


In [14]:
print(
    "Time range:",
    pd.Timestamp(model_data.time.values[0]).strftime("%Y-%m"),
    "to",
    pd.Timestamp(model_data.time.values[-1]).strftime("%Y-%m")
)

print(
    "Number of months:",
    model_data.sizes["time"]
)

print(
    "Grid size:",
    model_data.sizes["y"],
    "x",
    model_data.sizes["x"]
)

print(
    "Variables:",
    list(model_data.data_vars)
)

Time range: 2016-01 to 2025-12
Number of months: 120
Grid size: 216 x 180
Variables: ['NDVI', 'EVI', 'NDRE', 'VV', 'VH', 'FVC', 'spatial_ref']


## 2. Existing Model Validation Results

Spatial block cross-validation was previously used to evaluate model performance while reducing spatial dependence between training and testing observations.

The saved cross-validation results are loaded directly rather than repeating model training. Five-fold spatial cross-validation results are available for both the 1-year and 5-year prediction horizons.

In [15]:
cv_1yr = pd.read_csv(
    "spatial_cv_1year_results.csv"
)

cv_5yr = pd.read_csv(
    "spatial_cv_5year_results.csv"
)

print("1-year CV results:")
display(cv_1yr)

print("\n5-year CV results:")
display(cv_5yr)

1-year CV results:


,Fold,Optical_MAE,SAR_MAE,Optical_RMSE,SAR_RMSE,Optical_R2,SAR_R2,MAE_change_pct,RMSE_change_pct,R2_change
0,1,8.119172,7.862631,11.246691,10.937165,0.363293,0.397857,3.159691,2.752148,0.034564
1,2,8.232603,7.971140,11.112639,10.740296,0.315000,0.360134,3.175939,3.350623,0.045135
2,3,8.185002,7.863691,11.347259,10.888757,0.217512,0.279470,3.925598,4.040642,0.061958
3,4,9.335084,9.041102,12.838086,12.425967,0.297619,0.341989,3.149216,3.210128,0.044371
4,5,9.946138,9.670792,13.913528,13.493901,0.268959,0.312390,2.768378,3.015963,0.043431



5-year CV results:


,Fold,Optical_MAE,SAR_MAE,Optical_RMSE,SAR_RMSE,Optical_R2,SAR_R2,MAE_change_pct,RMSE_change_pct,R2_change
0,1,8.334692,8.092445,11.912147,11.561434,0.107973,0.159725,2.906497,2.944158,0.051752
1,2,8.784159,8.509105,12.593491,12.110930,0.137304,0.202151,3.131249,3.831832,0.064847
2,3,8.531447,8.296184,12.005228,11.727031,0.141642,0.180963,2.757605,2.317304,0.039321
3,4,7.287790,7.087642,10.221965,9.950341,0.126031,0.171861,2.746345,2.657260,0.045830
4,5,7.974378,7.698378,11.266262,10.858452,-0.050454,0.024217,3.461081,3.619749,0.074671


In [16]:
print("1-year spatial CV")
print("-----------------")
print(
    f"Mean MAE change: "
    f"{cv_1yr['MAE_change_pct'].mean():+.2f}%"
)
print(
    f"Mean RMSE change: "
    f"{cv_1yr['RMSE_change_pct'].mean():+.2f}%"
)
print(
    f"Mean R² change: "
    f"{cv_1yr['R2_change'].mean():+.3f}"
)
print(
    "Folds with lower RMSE after adding SAR:",
    int(
        (
            cv_1yr["SAR_RMSE"]
            < cv_1yr["Optical_RMSE"]
        ).sum()
    ),
    "/ 5"
)


print("\n5-year spatial CV")
print("-----------------")
print(
    f"Mean MAE change: "
    f"{cv_5yr['MAE_change_pct'].mean():+.2f}%"
)
print(
    f"Mean RMSE change: "
    f"{cv_5yr['RMSE_change_pct'].mean():+.2f}%"
)
print(
    f"Mean R² change: "
    f"{cv_5yr['R2_change'].mean():+.3f}"
)
print(
    "Folds with lower RMSE after adding SAR:",
    int(
        (
            cv_5yr["SAR_RMSE"]
            < cv_5yr["Optical_RMSE"]
        ).sum()
    ),
    "/ 5"
)

1-year spatial CV
-----------------
Mean MAE change: +3.24%
Mean RMSE change: +3.27%
Mean R² change: +0.046
Folds with lower RMSE after adding SAR: 5 / 5

5-year spatial CV
-----------------
Mean MAE change: +3.00%
Mean RMSE change: +3.07%
Mean R² change: +0.055
Folds with lower RMSE after adding SAR: 5 / 5


## 3. Prepare Data for Post-model Analysis

Additional spatial visualisation and feature-importance analysis require reconstruction of selected model samples. The original study-area polygon is first rasterised onto the 30 m modelling grid.

Only the samples required for each subsequent analysis will be constructed, reducing memory use compared with retaining all prediction horizons and fitted models simultaneously.

In [17]:
with open("Study_site_output.json", "r") as f:
    study_json = json.load(f)

geometry = shape(
    study_json["features"][0]["geometry"]
)

study_area = gpd.GeoDataFrame(
    geometry=[geometry],
    crs="EPSG:7856"
)

x_res = abs(
    model_data.x.values[1]
    - model_data.x.values[0]
)

y_res = abs(
    model_data.y.values[1]
    - model_data.y.values[0]
)

transform = Affine(
    x_res,
    0,
    model_data.x.values[0] - x_res / 2,
    0,
    -y_res,
    model_data.y.values[0] + y_res / 2
)

study_mask_array = geometry_mask(
    [study_area.geometry.iloc[0]],
    out_shape=(
        model_data.sizes["y"],
        model_data.sizes["x"]
    ),
    transform=transform,
    invert=True
)

study_mask_30m = xr.DataArray(
    study_mask_array,
    coords={
        "y": model_data.y,
        "x": model_data.x
    },
    dims=("y", "x"),
    name="study_area_mask"
)

print(
    "Study-area pixels:",
    int(study_mask_30m.sum())
)

Study-area pixels: 28145


In [18]:
model_data_masked = model_data.where(
    study_mask_30m
)

print("Post-model analysis data ready.")
print(
    "Study-area pixels:",
    int(study_mask_30m.sum())
)
print(
    "Dataset period:",
    pd.Timestamp(model_data.time.values[0]).strftime("%Y-%m"),
    "to",
    pd.Timestamp(model_data.time.values[-1]).strftime("%Y-%m")
)

Post-model analysis data ready.
Study-area pixels: 28145
Dataset period: 2016-01 to 2025-12


### Reconstruct 1-Year Prediction Samples

For the 1-year prediction horizon, EO observations at month *t* are paired with FVC observations 12 months later.

Only observations with complete NDVI, EVI, NDRE, VV, VH and target FVC values are retained. The resulting dataset is used to reconstruct the spatial validation structure required for subsequent model diagnostics and mapping.

In [19]:
available_dates = pd.DatetimeIndex(
    model_data_masked.time.values
)

available_date_set = set(available_dates)


def create_date_pairs(dates, horizon_months):
    pairs = []

    for predictor_date in dates:

        target_date = (
            predictor_date
            + pd.DateOffset(months=horizon_months)
        )

        if target_date in available_date_set:
            pairs.append(
                (predictor_date, target_date)
            )

    return pairs


pairs_1yr = create_date_pairs(
    available_dates,
    12
)

print(
    "1-year date pairs:",
    len(pairs_1yr)
)

print(
    "First pair:",
    pairs_1yr[0]
)

print(
    "Last pair:",
    pairs_1yr[-1]
)

1-year date pairs: 108
First pair: (Timestamp('2016-01-01 00:00:00'), Timestamp('2017-01-01 00:00:00'))
Last pair: (Timestamp('2024-12-01 00:00:00'), Timestamp('2025-12-01 00:00:00'))


In [20]:
def build_ml_samples(
    dataset,
    date_pairs
):
    sample_frames = []

    for predictor_date, target_date in date_pairs:

        predictors = dataset[
            ["NDVI", "EVI", "NDRE", "VV", "VH"]
        ].sel(
            time=predictor_date
        )

        target = dataset["FVC"].sel(
            time=target_date
        )

        sample = pd.DataFrame({
            "x": np.tile(
                dataset.x.values,
                dataset.sizes["y"]
            ),
            "y": np.repeat(
                dataset.y.values,
                dataset.sizes["x"]
            ),
            "NDVI": predictors["NDVI"].values.ravel(),
            "EVI": predictors["EVI"].values.ravel(),
            "NDRE": predictors["NDRE"].values.ravel(),
            "VV": predictors["VV"].values.ravel(),
            "VH": predictors["VH"].values.ravel(),
            "FVC_target": target.values.ravel()
        })

        sample["predictor_date"] = predictor_date
        sample["target_date"] = target_date

        sample = sample.dropna(
            subset=[
                "NDVI",
                "EVI",
                "NDRE",
                "VV",
                "VH",
                "FVC_target"
            ]
        )

        sample_frames.append(sample)

    samples = pd.concat(
        sample_frames,
        ignore_index=True
    )

    return samples

In [21]:
samples_1yr = build_ml_samples(
    model_data_masked,
    pairs_1yr
)

print("1-year samples")
print("----------------")

print(
    "Samples:",
    len(samples_1yr)
)

print(
    "Predictor dates:",
    samples_1yr[
        "predictor_date"
    ].nunique()
)

print(
    "Target dates:",
    samples_1yr[
        "target_date"
    ].nunique()
)

print(
    "Unique spatial locations:",
    samples_1yr[
        ["x", "y"]
    ].drop_duplicates().shape[0]
)

1-year samples
----------------
Samples: 2431453
Predictor dates: 105
Target dates: 105
Unique spatial locations: 24224


### Reconstruct Spatial Blocks

Samples are assigned to 500 m spatial blocks using their projected coordinates. These blocks reproduce the spatial grouping used during model validation.

The following step reconstructs the five-fold spatial cross-validation structure. No model is trained at this stage.

In [ ]:
def add_spatial_blocks(
    samples,
    block_size=500
):
    samples["block_x"] = (
        samples["x"] // block_size
    ).astype("int32")

    samples["block_y"] = (
        samples["y"] // block_size
    ).astype("int32")

    samples["block_id"] = (
        samples["block_x"].astype(str)
        + "_"
        + samples["block_y"].astype(str)
    )

    return samples


samples_1yr = add_spatial_blocks(
    samples_1yr,
    block_size=500
)

print(
    "1-year spatial blocks:",
    samples_1yr["block_id"].nunique()
)

### Reconstruct the Spatial Cross-validation Folds

The original validation used five-fold GroupKFold cross-validation, with the 500 m spatial block identifier used as the grouping variable.

This ensures that observations belonging to the same spatial block are not divided between the training and testing sets within a fold.

In [ ]:
gkf = GroupKFold(
    n_splits=5
)

groups_1yr = samples_1yr[
    "block_id"
]

fold_structure_1yr = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        samples_1yr,
        groups=groups_1yr
    ),
    start=1
):

    train_blocks = samples_1yr.iloc[
        train_idx
    ]["block_id"].nunique()

    test_blocks = samples_1yr.iloc[
        test_idx
    ]["block_id"].nunique()

    fold_structure_1yr.append({
        "Fold": fold,
        "Training blocks": train_blocks,
        "Testing blocks": test_blocks,
        "Training observations": len(train_idx),
        "Testing observations": len(test_idx)
    })


fold_structure_1yr = pd.DataFrame(
    fold_structure_1yr
)

display(fold_structure_1yr)

## 4. Spatial Cross-validation Results

Model performance across the five spatial cross-validation folds was compared between the optical-only and optical-plus-SAR configurations.

RMSE is used to compare prediction error across folds, while R² indicates the proportion of variation in future FVC explained by each model.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.5),
    sharey=True
)

folds = cv_1yr["Fold"]

axes[0].plot(
    folds,
    cv_1yr["Optical_RMSE"],
    marker="o",
    label="Optical-only"
)

axes[0].plot(
    folds,
    cv_1yr["SAR_RMSE"],
    marker="o",
    label="Optical + SAR"
)

axes[0].set_title("1-Year Prediction")
axes[0].set_xlabel("Spatial CV Fold")
axes[0].set_ylabel("RMSE (FVC percentage points)")
axes[0].set_xticks(folds)
axes[0].legend()


axes[1].plot(
    folds,
    cv_5yr["Optical_RMSE"],
    marker="o",
    label="Optical-only"
)

axes[1].plot(
    folds,
    cv_5yr["SAR_RMSE"],
    marker="o",
    label="Optical + SAR"
)

axes[1].set_title("5-Year Prediction")
axes[1].set_xlabel("Spatial CV Fold")
axes[1].set_xticks(folds)
axes[1].legend()

fig.suptitle(
    "Prediction Error Across Spatial Cross-Validation Folds",
    fontsize=14
)

plt.tight_layout()
plt.show()

Across all five spatial folds, the optical-plus-SAR model produced lower RMSE than the optical-only model for both prediction horizons. The magnitude of the improvement varied between spatial folds, but the direction of the change was consistent.

The mean RMSE reduction associated with adding Sentinel-1 SAR was approximately 3.27% for the 1-year prediction and 3.07% for the 5-year prediction.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.5),
    sharey=True
)

axes[0].plot(
    cv_1yr["Fold"],
    cv_1yr["Optical_R2"],
    marker="o",
    label="Optical-only"
)

axes[0].plot(
    cv_1yr["Fold"],
    cv_1yr["SAR_R2"],
    marker="o",
    label="Optical + SAR"
)

axes[0].set_title("1-Year Prediction")
axes[0].set_xlabel("Spatial CV Fold")
axes[0].set_ylabel("R²")
axes[0].set_xticks(cv_1yr["Fold"])
axes[0].axhline(0, linewidth=1)
axes[0].legend()


axes[1].plot(
    cv_5yr["Fold"],
    cv_5yr["Optical_R2"],
    marker="o",
    label="Optical-only"
)

axes[1].plot(
    cv_5yr["Fold"],
    cv_5yr["SAR_R2"],
    marker="o",
    label="Optical + SAR"
)

axes[1].set_title("5-Year Prediction")
axes[1].set_xlabel("Spatial CV Fold")
axes[1].set_xticks(cv_5yr["Fold"])
axes[1].axhline(0, linewidth=1)
axes[1].legend()

fig.suptitle(
    "Predictive Performance Across Spatial Cross-Validation Folds",
    fontsize=14
)

plt.tight_layout()
plt.show()

The 1-year models showed substantially stronger predictive performance than the 5-year models. Adding SAR increased R² in every spatial fold for both prediction horizons.

For the 1-year prediction, the mean R² increase was approximately 0.046. For the 5-year prediction, the mean increase was approximately 0.055. However, the overall R² values of the 5-year models remained relatively low, indicating that current EO conditions have limited ability to explain FVC five years into the future.

## 4. Spatial Cross-validation Results

Model performance across the five spatial cross-validation folds was compared between the optical-only and optical-plus-SAR configurations.

RMSE is used to compare prediction error across folds, while R² indicates the proportion of variation in future FVC explained by each model.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.5),
    sharey=True
)

folds = cv_1yr["Fold"]

axes[0].plot(
    folds,
    cv_1yr["Optical_RMSE"],
    marker="o",
    label="Optical-only"
)

axes[0].plot(
    folds,
    cv_1yr["SAR_RMSE"],
    marker="o",
    label="Optical + SAR"
)

axes[0].set_title("1-Year Prediction")
axes[0].set_xlabel("Spatial CV Fold")
axes[0].set_ylabel("RMSE (FVC percentage points)")
axes[0].set_xticks(folds)
axes[0].legend()


axes[1].plot(
    folds,
    cv_5yr["Optical_RMSE"],
    marker="o",
    label="Optical-only"
)

axes[1].plot(
    folds,
    cv_5yr["SAR_RMSE"],
    marker="o",
    label="Optical + SAR"
)

axes[1].set_title("5-Year Prediction")
axes[1].set_xlabel("Spatial CV Fold")
axes[1].set_xticks(folds)
axes[1].legend()

fig.suptitle(
    "Contribution of Sentinel-1 SAR to Prediction Accuracy",
    fontsize=14
)

plt.tight_layout()
plt.show()

### Contribution of Sentinel-1 SAR

The relative reduction in RMSE after adding Sentinel-1 SAR is compared across the five spatial cross-validation folds for both prediction horizons.

Positive values indicate lower prediction errors when SAR variables are included.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(1, 6)
width = 0.35

ax.bar(
    x - width / 2,
    cv_1yr["RMSE_change_pct"],
    width,
    label="1-Year"
)

ax.bar(
    x + width / 2,
    cv_5yr["RMSE_change_pct"],
    width,
    label="5-Year"
)

ax.axhline(0, color="black", linewidth=0.8)

ax.axhline(
    cv_1yr["RMSE_change_pct"].mean(),
    color="tab:blue",
    linestyle="--",
    linewidth=1,
    label="1-Year Mean"
)

ax.axhline(
    cv_5yr["RMSE_change_pct"].mean(),
    color="tab:orange",
    linestyle="--",
    linewidth=1,
    label="5-Year Mean"
)

ax.set_xlabel("Spatial CV Fold")
ax.set_ylabel("RMSE Reduction (%)")
ax.set_title("Contribution of Sentinel-1 SAR")

ax.set_xticks(x)
ax.legend()

plt.tight_layout()

plt.savefig(
    "SAR_RMSE_Contribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 5. Vegetation Recovery Trajectories (2016–2025)

Monthly Fractional Vegetation Cover (FVC) is examined to characterise vegetation dynamics within the New Acland study area.

FVC is calculated as the sum of photosynthetic vegetation (PV) and non-photosynthetic vegetation (NPV). Monthly mean FVC is calculated across valid pixels within the study-area boundary.

The resulting time series provides context for interpreting the machine-learning predictions.

In [ ]:
# Calculate spatial mean FVC for each month
fvc_monthly_mean = (
    model_data_masked["FVC"]
    .mean(dim=["x", "y"], skipna=True)
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    fvc_monthly_mean.index,
    fvc_monthly_mean.values,
    linewidth=1.5,
    label="Monthly Mean FVC"
)

ax.plot(
    fvc_monthly_mean.rolling(
        window=12,
        center=True,
        min_periods=6
    ).mean(),
    linewidth=2.5,
    label="12-Month Rolling Mean"
)

ax.set_title(
    "Vegetation Cover Dynamics at New Acland Mine (2016–2025)"
)

ax.set_xlabel("Year")
ax.set_ylabel("Mean FVC (%)")
ax.set_ylim(0, 100)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "FVC_Monthly_Trajectory.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fvc_annual = (
    fvc_monthly_mean
    .resample("YS")
    .mean()
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    fvc_annual.index.year,
    fvc_annual.values,
    marker="o",
    linewidth=2
)

ax.set_title(
    "Annual Mean Fractional Vegetation Cover (2016–2025)"
)

ax.set_xlabel("Year")
ax.set_ylabel("Mean FVC (%)")
ax.set_xticks(fvc_annual.index.year)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    "FVC_Annual_Trajectory.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(fvc_annual.round(2))

## 6. Spatial Distribution of FVC Change

Annual mean FVC maps are compared between 2016 and 2025 to investigate spatial patterns of vegetation-cover change.

The difference is calculated as:

$$
\Delta FVC = FVC_{2025} - FVC_{2016}
$$

Positive values indicate an increase in vegetation cover, while negative values indicate a decrease.

These changes may reflect rehabilitation, seasonal and climatic influences, or other land-cover processes.

In [ ]:
fvc_2016 = (
    model_data_masked["FVC"]
    .sel(time=slice("2016-01", "2016-12"))
    .mean(dim="time", skipna=True)
)

fvc_2025 = (
    model_data_masked["FVC"]
    .sel(time=slice("2025-01", "2025-12"))
    .mean(dim="time", skipna=True)
)

fvc_change = fvc_2025 - fvc_2016

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
    gridspec_kw={"wspace":0.35}
)

im1 = fvc_2016.plot(
    ax=axes[0],
    vmin=0,
    vmax=100,
    cmap="YlGn",
    add_colorbar=False
)

axes[0].set_title("Mean FVC — 2016")

fvc_2025.plot(
    ax=axes[1],
    vmin=0,
    vmax=100,
    cmap="YlGn",
    add_colorbar=False
)

axes[1].set_title("Mean FVC — 2025")

im3 = fvc_change.plot(
    ax=axes[2],
    vmin=-50,
    vmax=50,
    cmap="RdYlGn",
    add_colorbar=False
)

axes[2].set_title("FVC Change (2025 − 2016)")

fig.colorbar(
    im1,
    ax=axes[:2],
    label="FVC (%)",
    shrink=0.75,
    pad=0.04
)

fig.colorbar(
    im3,
    ax=axes[2],
    label="FVC Change (percentage points)",
    shrink=0.75,
    pad=0.04
)

plt.savefig(
    "FVC_Spatial_Change.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 7. Random Forest Feature Importance

Feature importance is used to examine the relative influence of NDVI, EVI, NDRE, VV and VH on FVC prediction.

Impurity-based Random Forest feature importance provides a descriptive measure of predictor usage. Because vegetation indices may be correlated, these values should not be interpreted as independent causal effects.

Feature importance is compared between the 1-year and 5-year prediction horizons.

The relative importance of Sentinel-2 vegetation indices and Sentinel-1 SAR variables was compared between the 1-year and 5-year Random Forest models.

Feature importance values describe the relative contribution of each predictor to the fitted Random Forest models. However, these values should be interpreted cautiously because correlated predictors may share predictive information.

The independent optical-only versus optical-plus-SAR comparison provides additional evidence of the contribution of Sentinel-1 SAR to prediction performance.

In [ ]:
importance_1yr = pd.read_csv(
    "feature_importance_1year.csv"
)

importance_5yr = pd.read_csv(
    "feature_importance_5year.csv"
)

importance_comparison = (
    importance_1yr
    .merge(
        importance_5yr,
        on="Feature",
        suffixes=("_1yr", "_5yr")
    )
    .set_index("Feature")
    .reindex(["NDVI", "EVI", "NDRE", "VV", "VH"])
)

display(importance_comparison.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

importance_comparison[
    ["Importance_1yr", "Importance_5yr"]
].plot(
    kind="bar",
    ax=ax,
    width=0.7
)

ax.set_title(
    "Random Forest Feature Importance by Prediction Horizon"
)

ax.set_xlabel("Predictor Variable")
ax.set_ylabel("Relative Feature Importance")

ax.legend(["1-Year", "5-Year"])

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    "Feature_Importance_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 8. Spatial Prediction and Error Maps

Spatial prediction maps are used to examine the geographical distribution of model predictions and errors.

Observed and predicted FVC are compared using the optical-plus-SAR Random Forest models.

Prediction error is calculated as:

$$
Error = Predicted\ FVC - Observed\ FVC
$$

Positive values indicate overprediction, while negative values indicate underprediction.

Only observations from the independent spatial test datasets are used.

In [ ]:
predictions_1yr = pd.read_csv(
    "predictions_1year_test.csv",
    parse_dates=["predictor_date", "target_date"]
)

predictions_5yr = pd.read_csv(
    "predictions_5year_test.csv",
    parse_dates=["predictor_date", "target_date"]
)

print("1-year observations:", len(predictions_1yr))
print("5-year observations:", len(predictions_5yr))

In [ ]:
def plot_spatial_predictions(
    predictions,
    target_month,
    horizon_label
):

    target_month = pd.Timestamp(target_month)

    map_data = predictions.loc[
        predictions["target_date"] == target_month,
        [
            "x",
            "y",
            "FVC_target",
            "FVC_pred_optical_sar"
        ]
    ].copy()

    if map_data.empty:
        print(f"No observations for {target_month:%Y-%m}")
        return

    map_data["Error"] = (
        map_data["FVC_pred_optical_sar"]
        - map_data["FVC_target"]
    )

    observed_grid = (
        map_data
        .pivot(index="y", columns="x", values="FVC_target")
        .sort_index(ascending=False)
    )

    predicted_grid = (
        map_data
        .pivot(index="y", columns="x", values="FVC_pred_optical_sar")
        .sort_index(ascending=False)
    )

    error_grid = (
        map_data
        .pivot(index="y", columns="x", values="Error")
        .sort_index(ascending=False)
    )

    extent = [
        observed_grid.columns.min(),
        observed_grid.columns.max(),
        observed_grid.index.min(),
        observed_grid.index.max()
    ]

    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 5),
        gridspec_kw={"wspace": 0.4}
    )

    im1 = axes[0].imshow(
        observed_grid.values,
        extent=extent,
        origin="upper",
        vmin=0,
        vmax=100,
        cmap="YlGn"
    )

    axes[0].set_title("Observed FVC")

    axes[1].imshow(
        predicted_grid.values,
        extent=extent,
        origin="upper",
        vmin=0,
        vmax=100,
        cmap="YlGn"
    )

    axes[1].set_title("Predicted FVC")

    im3 = axes[2].imshow(
        error_grid.values,
        extent=extent,
        origin="upper",
        vmin=-30,
        vmax=30,
        cmap="RdBu_r"
    )

    axes[2].set_title("Prediction Error")

    for ax in axes:
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")
        ax.set_aspect("equal")

    fig.colorbar(
        im1,
        ax=axes[:2],
        label="FVC (%)",
        shrink=0.75,
        pad=0.04
    )

    fig.colorbar(
        im3,
        ax=axes[2],
        label="Error (percentage points)",
        shrink=0.75,
        pad=0.04
    )

    fig.suptitle(
        f"{horizon_label} Spatial Prediction — {target_month:%B %Y}",
        fontsize=14
    )

    filename = (
        f"Spatial_Prediction_{horizon_label.replace('-', '')}_"
        f"{target_month:%Y%m}.png"
    )

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Map observations:", len(map_data))
    print("Mean prediction error:", round(map_data["Error"].mean(), 2))

In [ ]:
# 1-year Spatial Map
plot_spatial_predictions(
    predictions_1yr,
    target_month="2024-06-01",
    horizon_label="1-Year"
)

In [ ]:
# 5-year Spatial Map
plot_spatial_predictions(
    predictions_5yr,
    target_month="2024-06-01",
    horizon_label="5-Year"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

Path("Report_Figures").mkdir(exist_ok=True)

models = [
    "1-Year\nOptical",
    "1-Year\nOptical + SAR",
    "5-Year\nOptical",
    "5-Year\nOptical + SAR"
]

metrics = {
    "MAE": [7.995, 7.757, 7.440, 7.265],
    "RMSE": [11.173, 10.891, 10.696, 10.474],
    "R²": [0.316, 0.350, 0.069, 0.107]
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

colors = ["#A9C7E8", "#2864A0", "#F5C49B", "#D97831"]

for ax, (metric, values) in zip(axes, metrics.items()):

    bars = ax.bar(
        models,
        values,
        color=colors,
        edgecolor="black",
        linewidth=0.5
    )

    ax.set_title(metric, fontsize=13, fontweight="bold")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    ax.set_ylim(0, max(values) * 1.2)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.025,
            f"{value:.3f}",
            ha="center",
            fontsize=10
        )

fig.suptitle(
    "Random Forest Model Performance Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "Report_Figures/Model_Performance_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd

df_1yr = pd.read_csv("predictions_1year_test.csv")
df_5yr = pd.read_csv("predictions_5year_test.csv")

print("1-Year columns:")
print(df_1yr.columns.tolist())

print("\n5-Year columns:")
print(df_5yr.columns.tolist())

print("\n1-Year preview:")
display(df_1yr.head())

print("\n5-Year preview:")
display(df_5yr.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score

output_dir = Path("Report_Figures")
output_dir.mkdir(exist_ok=True)

datasets = [
    (df_1yr, "FVC_pred_optical_sar", "1-Year Optical + SAR"),
    (df_5yr, "FVC_pred_optical", "5-Year Optical-only"),
    (df_5yr, "FVC_pred_optical_sar", "5-Year Optical + SAR")
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5),
    constrained_layout=True
)

for ax, (df, pred_col, title) in zip(axes, datasets):

    data = df[["FVC_target", pred_col]].dropna()

    observed = data["FVC_target"].to_numpy()
    predicted = data[pred_col].to_numpy()

    rmse = np.sqrt(mean_squared_error(observed, predicted))
    r2 = r2_score(observed, predicted)

    hb = ax.hexbin(
        observed,
        predicted,
        gridsize=55,
        cmap="viridis",
        mincnt=1,
        bins="log"
    )

    ax.plot(
        [0, 100],
        [0, 100],
        "r--",
        linewidth=1.5,
        label="1:1 Line"
    )

    ax.set_xlim(0, 105)
    ax.set_ylim(0, 105)

    ax.set_aspect("equal", adjustable="box")

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Observed FVC (%)")
    ax.set_ylabel("Predicted FVC (%)")

    ax.text(
        0.95,
        0.05,
        f"RMSE = {rmse:.2f}\nR² = {r2:.3f}",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=10,
        bbox=dict(
            facecolor="white",
            edgecolor="gray",
            alpha=0.9
        )
    )

    fig.colorbar(
        hb,
        ax=ax,
        shrink=0.7,
        label="Log-scaled pixel count"
    )

fig.suptitle(
    "Observed vs Predicted Fractional Vegetation Cover",
    fontsize=15,
    fontweight="bold"
)

plt.savefig(
    output_dir / "Observed_vs_Predicted_FVC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()